# PhysRevD 78, 123006 (2008) — Self-Contained Implementation

**Kahniashvili, Campanelli, Gogoberidze, Maravin, Ratra (2008)**  
*Gravitational radiation from primordial helical inverse cascade MHD turbulence*

---

## Structure of this notebook

1. **Physical constants & cosmological relations** (Eqs. 11, 16, 53)
2. **Phase-transition parameters** — κ, v_b, v_0 (Eqs. 13–15)
3. **Turbulence parameters** — l_0, k_0, M, τ_0, ζ_*, γ
4. **Model A inverse cascade** — ξ_M, E_M, k_S, η(k), H_{ijij} (Eqs. 29–35, 45)
5. **Model B inverse cascade** — ξ_M, E_M, k_S, η(k), H_{ijij} (Eqs. 36–41, 47)
6. **Direct-cascade (first-stage) H_{ijij}** (Eqs. 27–28, 43)
7. **GW energy density Ω_GW(ω)** (Eq. 49)
8. **Strain amplitude h_C(f)** (Eqs. 50–51)
9. **Plots** — reproduce Figs. 1 & 2 style (h_C vs f, no sensitivity curves)

Natural units: ħ = c = k_B = 1 throughout.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate, special

plt.rcParams.update({'figure.dpi': 110, 'font.size': 13})

## 1. Physical constants & cosmological relations

In [ ]:
# ── Fundamental constants ─────────────────────────────────────────────────────
h0       = 0.67            # dimensionless Hubble parameter
H0_Hz    = h0 * 1e5 / 3.086e22   # H_0 in Hz ≈ 2.17e-18 Hz

# ── Standard-model relativistic degrees of freedom ───────────────────────────
g_star_SM = 106.75

# ─────────────────────────────────────────────────────────────────────────────
# Eq. (16)  H_*^2 = 8π^3 G T_*^4 / 90  (including g_* via radiation density)
#
# Standard result (Kolb & Turner):  H_* ≈ 1.66 g_*^{1/2} T_*^2 / M_Pl
# M_Pl = 1.22e19 GeV, 1 GeV ↔ 1.519e24 Hz (via ħ = 6.582e-25 GeV·s)
#
# Numerically:  H_*(T, g) ≈ 2.07e10 × (T/100 GeV)^2 × (g/100)^{1/2}  Hz

def H_star_Hz(T_GeV: float, g_star: float = g_star_SM) -> float:
    """Hubble rate at phase transition in Hz.  Eq. (16).
    H_* = 1.66 sqrt(g_*) T_*^2 / M_Pl, converted to Hz."""
    M_Pl_GeV = 1.22e19           # non-reduced Planck mass in GeV
    GeV_to_Hz = 1.0 / 6.582e-25  # 1 GeV in Hz  (= 1/ħ)
    H_GeV = 1.66 * g_star**0.5 * T_GeV**2 / M_Pl_GeV
    return H_GeV * GeV_to_Hz

# ─────────────────────────────────────────────────────────────────────────────
# Eq. (11)  a_* / a_0  ≈  8e-16 (100 GeV / T_*) (100 / g_*)^{1/3}

def scale_factor_ratio(T_GeV: float, g_star: float = g_star_SM) -> float:
    """a_* / a_0.  Eq. (11)."""
    return 8e-16 * (100.0 / T_GeV) * (100.0 / g_star)**(1.0/3.0)

# ─────────────────────────────────────────────────────────────────────────────
# Eq. (53)  f_H ≡ H_* (a_*/a_0)  ≈  1.6e-5 Hz (g_*/100)^{1/6} (T_*/100 GeV)

def f_H_Hz(T_GeV: float, g_star: float = g_star_SM) -> float:
    """Characteristic (Hubble) frequency today in Hz.  Eq. (53)."""
    return 1.6e-5 * (g_star / 100.0)**(1.0/6.0) * (T_GeV / 100.0)

# Quick self-check
T_test, g_test = 100.0, 106.75
Hstar_test = H_star_Hz(T_test, g_test)
astar_test  = scale_factor_ratio(T_test, g_test)
fH_test     = f_H_Hz(T_test, g_test)
print(f"H_*   = {Hstar_test:.3e} Hz  (expect ~2e10)")
print(f"a*/a0 = {astar_test:.3e}")
print(f"f_H   = {fH_test:.3e} Hz  (expect ~1.6e-5)")
print(f"f_H check  H_* * a*/a0 = {Hstar_test * astar_test:.3e} Hz")
print(f"H_0   = {H0_Hz:.3e} Hz")


## 2. Phase-transition parameters

$$
\kappa(\alpha) = \frac{1}{1+0.715\alpha}\left[0.715\alpha + \frac{4}{27}\left(\frac{3\alpha}{2}\right)^{1/2}\right], \quad\text{Eq. (13)}
$$
$$
v_b(\alpha) = \frac{1/\sqrt{3}+(\alpha^2+2\alpha/3)^{1/2}}{1+\alpha}, \quad\text{Eq. (14)}
$$
$$
v_0(\alpha) = \left(\frac{3\kappa\alpha}{4+3\kappa\alpha}\right)^{1/2}, \quad\text{Eq. (15)}
$$

In [ ]:
def kappa(alpha: float) -> float:
    """Efficiency factor.  Eq. (13)."""
    return (0.715*alpha + (4.0/27.0)*(1.5*alpha)**0.5) / (1.0 + 0.715*alpha)

def v_bubble(alpha: float) -> float:
    """Bubble wall velocity.  Eq. (14)."""
    return (1.0/np.sqrt(3.0) + np.sqrt(alpha**2 + 2.0*alpha/3.0)) / (1.0 + alpha)

def v_eddy(alpha: float) -> float:
    """Turbulent eddy velocity (Mach number).  Eq. (15)."""
    k = kappa(alpha)
    return np.sqrt(3.0*k*alpha / (4.0 + 3.0*k*alpha))

# Demo
alpha_demo = 0.5
print(f"α={alpha_demo}:  κ={kappa(alpha_demo):.4f}  v_b={v_bubble(alpha_demo):.4f}  v_0={v_eddy(alpha_demo):.4f}")

## 3. Turbulence parameters

The initial (largest) eddy scale is set by the bubble collision:
$$l_0 = v_b \beta^{-1}, \quad k_0 = 2\pi/l_0,$$
where $\beta^{-1}$ sets the timescale for the phase transition.

The Mach number $M = v_0$, and the initial turnover time $\tau_0 = l_0/v_0$.

The parameter $\gamma$ (number of eddy-lengths per Hubble radius):
$$\gamma = v_0 / (l_0 H_*) = v_0 \beta / (v_b H_*).$$

The initial helicity parameter $\zeta_*$ characterises how close to maximally helical the field is (Eq. 26). We keep it as a free parameter.

In [ ]:
def turb_params(alpha: float, beta_over_Hstar: float, T_GeV: float,
                g_star: float = g_star_SM):
    """
    Compute all turbulence parameters from phase-transition inputs.

    Parameters
    ----------
    alpha          : vacuum-energy / thermal-energy ratio
    beta_over_Hstar: β/H_*
    T_GeV          : phase-transition temperature in GeV
    g_star         : relativistic d.o.f.

    Returns dict (all wavenumbers in units of H_*, all times in units of H_*^{-1})
    """
    kap = kappa(alpha)
    vb  = v_bubble(alpha)
    v0  = v_eddy(alpha)
    M   = v0   # Mach number

    Hstar = H_star_Hz(T_GeV, g_star)   # Hz  (correct formula: ~2e10 Hz at 100 GeV)
    fH    = f_H_Hz(T_GeV, g_star)

    # γ ≡ l_0 H_* = v_b / (β/H_*)  (dimensionless, << 1 for β >> H_*)
    # Derived from τ_cas ≤ H_*^{-1}:  k0/kS ≤ (v0/γ)^{1/2} ζ_*^{1/4}
    gamma    = vb / beta_over_Hstar   # = l_0 * H_*
    l0       = gamma                   # alias
    k0       = 2.0*np.pi / l0          # k_0 in units of H_*
    tau0     = l0 / v0                 # τ_0 in units of H_*^{-1}

    return dict(kap=kap, vb=vb, v0=v0, M=M,
                k0=k0, l0=l0, tau0=tau0, gamma=gamma,
                Hstar_Hz=Hstar, fH_Hz=fH,
                T_GeV=T_GeV, g_star=g_star)

# Demo
p = turb_params(alpha=0.5, beta_over_Hstar=100.0, T_GeV=100.0)
for k, v in p.items():
    print(f"  {k:12s} = {v:.4e}")

zeta_demo = 0.15
print(f"\nγ = {p['gamma']:.4e},  M·ζ_*^{{1/2}} = {p['M']*zeta_demo**0.5:.4e}")
print("Inverse cascade (Model A) exists:", p['gamma'] <= p['M'] * zeta_demo**0.5)


## 4. Model A inverse cascade — H_{ijij} for second stage

From Eq. (33): the time-independent magnetic energy spectrum is
$$E_M(k) = \frac{C_1 v_1^2}{k_0}, \quad k_S < k < k_0.$$

The autocorrelation function (Eq. 34):
$$\eta(k) = \frac{\sqrt{2\pi}}{\tau_1}\left(\frac{k}{k_0}\right)^2$$

where $\tau_1 \approx \tau_0 / \zeta_*^{1/2}$ and $v_1 \approx \zeta_*^{1/2} v_0$.

The cascade stops at $k_S$ satisfying (Eq. 35):
$$\frac{k_0}{k_S} \leq \left(\frac{v_0}{\gamma}\right)^{1/2} \zeta_*^{1/4}$$

i.e. $k_S = k_0 \left(\frac{\gamma}{v_0}\right)^{1/2} \zeta_*^{-1/4}$.

The $H_{ijij}$ integral for Model A second stage (Eq. 45) after the aero-acoustic approximation $\mathbf{k}\to 0$:

$$H_{ijij}(0,\omega) = \frac{7C_1^2 M^3 \zeta_*^{3/2}}{12\pi^{3/2}k_0}
\int_{k_S}^{k_0}\frac{dk}{k^4}
\exp\!\left(-\frac{\omega^2 k_0^2}{\zeta_* M^2 k^3}\right)
\operatorname{erfc}\!\left(-\frac{\omega k_0}{\zeta_*^{1/2} M k^{3/2}}\right).$$

Here $C_1 \sim 1$ (order-unity constant). We set $C_1 = 1$.

In [ ]:
C1 = 1.0   # order-unity constant (paper states C_1 ~ 1)

def k_S_modelA(tp: dict, zeta_star: float) -> float:
    """
    Smallest wavenumber where the inverse cascade stops, Model A.
    Eq. (35):  k0/kS = (v0/gamma)^{1/2} * zeta_*^{1/4}
    Returns kS in units of H_*.
    """
    ratio = (tp['v0'] / tp['gamma'])**0.5 * zeta_star**0.25
    return tp['k0'] / ratio

def H_ijij_modelA(omega_Hstar: float, tp: dict, zeta_star: float,
                  n_pts: int = 400) -> float:
    """
    H_{ijij}(0, omega) for the second stage (Model A).
    Eq. (45).

    Parameters
    ----------
    omega_Hstar : angular frequency in units of H_*
    tp          : turbulence parameter dict from turb_params()
    zeta_star   : initial helicity parameter (< 1)

    Returns H (dimensionless, in units of H_*^{-1} × k_0 normalisation absorbed)
    """
    k0   = tp['k0']
    M    = tp['M']
    kS   = k_S_modelA(tp, zeta_star)

    if kS >= k0:
        return 0.0

    prefactor = 7.0 * C1**2 * M**3 * zeta_star**1.5 / (12.0 * np.pi**1.5 * k0)

    k_vals = np.geomspace(kS, k0, n_pts)
    arg_exp  = omega_Hstar**2 * k0**2 / (zeta_star * M**2 * k_vals**3)
    arg_erfc = -omega_Hstar * k0 / (np.sqrt(zeta_star) * M * k_vals**1.5)
    integrand = k_vals**(-4) * np.exp(-arg_exp) * special.erfc(arg_erfc)

    return prefactor * np.trapz(integrand, k_vals)


def tau_T_modelA(tp: dict, zeta_star: float) -> float:
    """
    Duration of turbulence for second stage, Model A.
    τ_T^{II} ~ τ_cas = τ_1 (k0/kS)^2,  τ_1 = τ_0 / sqrt(zeta_*)
    but the max cascade time is H_*^{-1}, so τ_T^{II} = min(τ_cas, H_*^{-1}).
    Returns in units of H_*^{-1}.
    """
    tau1    = tp['tau0'] / np.sqrt(zeta_star)  # τ_1, units H_*^{-1}
    kS      = k_S_modelA(tp, zeta_star)
    tau_cas = tau1 * (tp['k0'] / kS)**2        # Eq. (32)
    return min(tau_cas, 1.0)                   # cannot exceed H_*^{-1} = 1 (in these units)


# Sanity check
tp_demo = turb_params(0.5, 100.0, 100.0)
zeta_demo = 0.15
kS_A = k_S_modelA(tp_demo, zeta_demo)
print(f"k0 = {tp_demo['k0']:.3e} H_*,  kS (Model A) = {kS_A:.3e} H_*")
print(f"k0/kS = {tp_demo['k0']/kS_A:.2f}")
omega_test = 2.0*np.pi   # = omega_peak (Hubble freq)
H_A = H_ijij_modelA(omega_test, tp_demo, zeta_demo)
print(f"H_ijij (Model A, omega=2πH_*) = {H_A:.4e}")

## 5. Model B inverse cascade — H_{ijij} for second stage

Model B uses a different evolution law (Refs. [34,35]): $\xi_M \propto (1+t/\tau_1)^{2/3}$.

Cascade stops at (Eq. 41):
$$\frac{k_0}{k_S} \leq \left(\frac{v_0}{\gamma}\right)^{2/3}\zeta_*^{1/3}$$

Autocorrelation (Eq. 40):
$$\eta(k) = \frac{\sqrt{2\pi}}{\tau_1}\left(\frac{k}{k_0}\right)^{3/2}$$

$H_{ijij}$ for Model B second stage (Eq. 47):
$$H_{ijij}(0,\omega) = \frac{7C_1^2 M^3 \zeta_*^{3/2}}{6\pi^{3/2}k_0^{3/2}}
\int_{k_S}^{k_0}\frac{dk}{k^{7/2}}
\exp\!\left(-\frac{\omega^2 k_0}{\zeta_* M^2 k^3}\right)
\operatorname{erfc}\!\left(-\frac{\omega k_0^{1/2}}{\zeta_*^{1/2} M k^{3/2}}\right).$$

In [ ]:
def k_S_modelB(tp: dict, zeta_star: float) -> float:
    """
    Smallest wavenumber where inverse cascade stops, Model B.
    Eq. (41):  k0/kS = (v0/gamma)^{2/3} * zeta_*^{1/3}
    """
    ratio = (tp['v0'] / tp['gamma'])**(2.0/3.0) * zeta_star**(1.0/3.0)
    return tp['k0'] / ratio


def H_ijij_modelB(omega_Hstar: float, tp: dict, zeta_star: float,
                  n_pts: int = 400) -> float:
    """
    H_{ijij}(0, omega) for second stage (Model B).  Eq. (47).
    """
    k0  = tp['k0']
    M   = tp['M']
    kS  = k_S_modelB(tp, zeta_star)

    if kS >= k0:
        return 0.0

    prefactor = 7.0 * C1**2 * M**3 * zeta_star**1.5 / (6.0 * np.pi**1.5 * k0**1.5)

    k_vals   = np.geomspace(kS, k0, n_pts)
    arg_exp  = omega_Hstar**2 * k0 / (zeta_star * M**2 * k_vals**3)
    arg_erfc = -omega_Hstar * np.sqrt(k0) / (np.sqrt(zeta_star) * M * k_vals**1.5)
    integrand = k_vals**(-3.5) * np.exp(-arg_exp) * special.erfc(arg_erfc)

    return prefactor * np.trapz(integrand, k_vals)


def tau_T_modelB(tp: dict, zeta_star: float) -> float:
    """
    Duration of turbulence for second stage, Model B.
    τ_cas = τ_1 (k0/kS)^{3/2},  Eq. (39).  Capped at H_*^{-1} = 1.
    """
    tau1    = tp['tau0'] / np.sqrt(zeta_star)
    kS      = k_S_modelB(tp, zeta_star)
    tau_cas = tau1 * (tp['k0'] / kS)**1.5    # Eq. (39)
    return min(tau_cas, 1.0)


# Check Model B
kS_B = k_S_modelB(tp_demo, zeta_demo)
print(f"k0/kS (Model B) = {tp_demo['k0']/kS_B:.2f}")
H_B = H_ijij_modelB(omega_test, tp_demo, zeta_demo)
print(f"H_ijij (Model B, omega=2πH_*) = {H_B:.4e}")

## 6. Direct-cascade (first stage) H_{ijij}

During the first stage the turbulence follows the Kolmogorov spectrum (Eq. 27–28):
$$E_M(k) = C_K \varepsilon^{2/3} k^{-5/3}, \qquad \eta(k) = \frac{\varepsilon^{1/3}}{\sqrt{2\pi}} k^{2/3}.$$

The gravitational wave source integral (Eq. 43):
$$H_{ijij}(0,\omega) = \frac{7C_K^2 \varepsilon}{6\pi^{3/2} k_0}
\int_{k_0}^{k_d} \frac{dk}{k^6}
\exp\!\left(-\frac{\omega^2}{\varepsilon^{2/3}k^{4/3}}\right)
\operatorname{erfc}\!\left(-\frac{\omega}{\varepsilon^{1/3}k^{2/3}}\right).$$

Here $\varepsilon = k_0 v_0^3$ is the energy dissipation rate per unit enthalpy (in units of $H_*$),
$k_d \gg k_0$ is the dissipation wavenumber (we integrate to $k_d = 10^4 k_0$),
and the total kinetic+magnetic energy is $2 \times$ (magnetic part) due to equipartition.

The duration of the first stage is $\tau_{s0} = s_0 \tau_0$ with $s_0 \sim 3$–$5$  
(paper uses $s_0 \approx 2$ for the averaging, cf. text after Eq. 27).

Peak frequency (Eq. 44): $\omega^{(I)}_{\rm max} = k_0 M$.

In [ ]:
C_K = 1.0   # Kolmogorov constant ~ 1
s0  = 2.0   # first-stage duration = s0 * τ_0  (paper uses effective s0/2)

def H_ijij_direct(omega_Hstar: float, tp: dict,
                  kd_ratio: float = 1e4, n_pts: int = 600) -> float:
    """
    H_{ijij}(0, omega) for first stage (Kolmogorov / direct cascade).  Eq. (43).

    The factor of 2 for equipartition (kinetic + magnetic) is included.
    """
    k0   = tp['k0']
    v0   = tp['v0']
    eps  = k0 * v0**3          # energy dissipation rate (in H_* units)
    k_d  = kd_ratio * k0

    # Factor 2 for equipartition (kinetic + magnetic)
    prefactor = 2.0 * 7.0 * C_K**2 * eps / (6.0 * np.pi**1.5 * k0)

    k_vals   = np.geomspace(k0, k_d, n_pts)
    arg_exp  = omega_Hstar**2 / (eps**(2.0/3.0) * k_vals**(4.0/3.0))
    arg_erfc = -omega_Hstar   / (eps**(1.0/3.0) * k_vals**(2.0/3.0))
    integrand = k_vals**(-6) * np.exp(-arg_exp) * special.erfc(arg_erfc)

    return prefactor * np.trapz(integrand, k_vals)


def tau_T_direct(tp: dict) -> float:
    """Duration of first stage in units of H_*^{-1}.  τ_T^{(I)} = s0 * τ_0."""
    return s0 * tp['tau0']


# Check
H_dir = H_ijij_direct(tp_demo['k0'] * tp_demo['M'], tp_demo)  # at peak freq
print(f"H_ijij (direct cascade, at omega_peak) = {H_dir:.4e}")
print(f"Peak freq for direct cascade (in H_*): omega_I = k0*M = {tp_demo['k0']*tp_demo['M']:.3e} H_*")

## 7. GW energy density Ω_GW(ω)

Eq. (49): the total fractional GW energy density at the moment of emission
$$\Omega_{\rm GW}(\omega_*) \approx 105 \frac{H_*^2}{H_0^2}
\sum_m \tau_T^{(m)} H_{ijij}^{(m)}(0,\omega_*)$$

where the sum is over stages $m = I$ (direct cascade), $II$ (inverse cascade).

The current Hubble parameter $H_0 = h_0 \times 100\,\text{km s}^{-1}\text{Mpc}^{-1}$.

In [ ]:
# ── Eq. (51) derivation note ─────────────────────────────────────────────────
# The paper's Eq. (51) is obtained by combining Eqs. (49) and (50).
# In dimensionless units (all quantities in units of H_*), it reads:
#
#   h_C(f) = 2e-14 × (100 GeV/T_*) × (100/g_*)^{1/3}
#              × [ Σ_m  τ̂_T^{(m)} × ω̂ × Ĥ_{ijij}^{(m)}(ω̂) ]^{1/2}
#
# where:
#   ω̂       = ω / H_*  = 2π f / f_H         (dimensionless angular frequency)
#   τ̂_T^{(m)} = τ_T^{(m)} × H_*              (dimensionless duration)
#   Ĥ_{ijij}  = H_{ijij} × H_*               (dimensionless, from Eqs. 43/45/47
#                                              with k in units of H_*)
#
# The extra factor ω̂ in the sum comes from d Ω_GW / d ln ω  ∝  ω^2 H_{ijij}(ω),
# combined with h_C^2 ∝ Ω_GW(f)/f  ⇒  h_C^2 ∝ ω × H_{ijij}(ω).
#
# Verification at peak (ω̂=2π, Model B, ζ=0.15, T=100 GeV, α=0.5, β/H_*=100):
#   Ĥ_{ijij}(2π) ≈ 7.4e-14  →  h_C ≈ 2e-14 × sqrt(2π × 7.4e-14) ≈ 1.4e-20  ✓

def h_C_spectrum(f_array_Hz: np.ndarray, tp: dict, zeta_star: float,
                 model: str = 'A') -> np.ndarray:
    """
    Gravitational wave strain amplitude h_C(f) via Eq. (51).

    Parameters
    ----------
    f_array_Hz : current frequencies in Hz
    tp         : turbulence parameters dict from turb_params()
    zeta_star  : initial magnetic helicity parameter (0 < ζ_* < 1)
    model      : 'A' or 'B' for the inverse-cascade model

    Returns
    -------
    h_C : dimensionless strain amplitude array
    """
    fH      = tp['fH_Hz']
    T_GeV   = tp['T_GeV']
    g_star  = tp['g_star']

    omega_hat = 2.0 * np.pi * f_array_Hz / fH   # dimensionless ω/H_*

    tau_I  = tau_T_direct(tp)
    if model == 'A':
        tau_II = tau_T_modelA(tp, zeta_star)
        H_cascade = lambda om: H_ijij_modelA(om, tp, zeta_star)
    else:
        tau_II = tau_T_modelB(tp, zeta_star)
        H_cascade = lambda om: H_ijij_modelB(om, tp, zeta_star)

    total = np.zeros_like(omega_hat)
    for i, om in enumerate(omega_hat):
        H_I  = H_ijij_direct(om, tp)
        H_II = H_cascade(om)
        # ω̂ × Σ_m τ̂_T^{(m)} × Ĥ_{ijij}^{(m)}
        total[i] = om * (tau_I * H_I + tau_II * H_II)

    prefactor = 2e-14 * (100.0 / T_GeV) * (100.0 / g_star)**(1.0/3.0)
    return prefactor * np.sqrt(np.maximum(total, 0.0))


def h_C_unmagnetized(f_array_Hz: np.ndarray, tp: dict) -> np.ndarray:
    """First-stage (direct-cascade) only — zero helicity case."""
    fH    = tp['fH_Hz']
    T_GeV = tp['T_GeV']
    g_star= tp['g_star']
    omega_hat = 2.0 * np.pi * f_array_Hz / fH
    tau_I = tau_T_direct(tp)
    total = np.array([om * tau_I * H_ijij_direct(om, tp) for om in omega_hat])
    prefactor = 2e-14 * (100.0 / T_GeV) * (100.0 / g_star)**(1.0/3.0)
    return prefactor * np.sqrt(np.maximum(total, 0.0))


# ── Spot check at peak ────────────────────────────────────────────────────────
tp_check = turb_params(0.5, 100.0, 100.0, g_star=100.0)
zeta_check = 0.15
om_peak_II = 2.0 * np.pi  # inverse-cascade peak: ω̂ = 2π (f = f_H)
H_B_peak = H_ijij_modelB(om_peak_II, tp_check, zeta_check)
tau_B = tau_T_modelB(tp_check, zeta_check)
hC_peak_analytic = 2e-14 * 1.0 * 1.0 * np.sqrt(om_peak_II * tau_B * H_B_peak)
print(f"Model B, ζ=0.15, T=100 GeV, peak h_C (analytic) = {hC_peak_analytic:.3e}")
print(f"  (paper Fig.1 shows ~1-2 × 10^{{-20}} for Model B dashed lines)")


## 8. Strain amplitude h_C(f)

Eq. (50):
$$h_C(f) = 1.26\times10^{-18}\left(\frac{\text{Hz}}{f}\right)^{1/2}\left[h_0^2\,\Omega_{\rm GW}(f)\right]^{1/2}$$

Here the frequency today $f = (a_*/a_0)\,\omega_*/(2\pi)$, with $\omega_*$ the angular frequency at emission.

So $\omega_*(f) = 2\pi f / (a_*/a_0)$ in physical units, or in units of $H_*$:
$$\omega_{*,H}(f) = \frac{2\pi f}{f_H} \quad (\text{where } f_H = H_*(a_*/a_0)).$$

In [ ]:
# h_C_spectrum and h_C_unmagnetized are defined in the cell above.
# This cell verifies the frequency-to-omega_hat mapping.

tp_verify = turb_params(0.5, 100.0, 100.0, g_star=100.0)
fH_v = tp_verify['fH_Hz']
print(f"f_H = {fH_v:.4e} Hz")
print(f"At f = f_H:   omega_hat = {2*np.pi*fH_v/fH_v:.4f}  (should be 2π ≈ 6.28)")
print(f"Peak I  freq: f = (k0*M/(2π)) * f_H = {tp_verify['k0']*tp_verify['M']/(2*np.pi)*fH_v:.3e} Hz  (Eq. 52)")
print(f"Peak II freq: f = f_H = {fH_v:.3e} Hz  (Eq. 54)")


## 9. Plots — reproducing Figs. 1 & 2 style

Parameters as in Fig. 1 of the paper: $g_*=100$, $T_*=100\,\text{GeV}$, $\alpha=0.5$, $\beta=100H_*$.

Left panel: $\zeta_*=0.15$, Right panel: $\zeta_*=0.05$.  
No sensitivity curves (per user request).

> **Note on constants $C_K, C_1$:** The paper sets $C_K = C_1 = 1$ (order-unity). The amplitude can differ by factors of order unity relative to the paper's figures due to these and other $O(1)$ choices (factor of 7 from the geometric kernel trace, etc.).

In [ ]:
# ── Frequency grid (LISA band) ────────────────────────────────────────────────
f_array = np.geomspace(1e-6, 3e-3, 80)   # Hz

# ── Parameter sets matching Fig. 1 (T_* = 100 GeV) ───────────────────────────
alpha_fig = 0.5
beta_fig  = 100.0    # β / H_*
T_fig     = 100.0    # GeV
g_fig     = 100.0

tp_fig = turb_params(alpha_fig, beta_fig, T_fig, g_star=g_fig)
print("Turbulence parameters (Fig. 1 setup):")
for kk, vv in tp_fig.items():
    print(f"  {kk:12s} = {vv:.4e}")

fH = tp_fig['fH_Hz']
f_peak_I  = tp_fig['k0'] * tp_fig['M'] / (2*np.pi) * fH    # Eq. (52)
f_peak_II = fH                                                # Eq. (54)
print(f"\nPeak freq direct cascade: f_I  = {f_peak_I:.3e} Hz")
print(f"Peak freq inv. cascade:   f_II = {f_peak_II:.3e} Hz  (= f_H)")

zeta_vals = [0.15, 0.05]


In [ ]:
print("Computing unmagnetized spectrum (direct cascade only)...")
hC_unmag = h_C_unmagnetized(f_array, tp_fig)
print(f"  peak h_C (unmag) = {hC_unmag.max():.3e}")

results = {}   # (zeta, model) -> h_C array
for zeta in zeta_vals:
    for model in ['A', 'B']:
        print(f"Computing zeta={zeta}, Model {model}...")
        results[(zeta, model)] = h_C_spectrum(f_array, tp_fig, zeta, model=model)
        print(f"  peak h_C = {results[(zeta, model)].max():.3e}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax_idx, zeta in enumerate(zeta_vals):
    ax = axes[ax_idx]

    ax.loglog(f_array, hC_unmag * 1e20,              'k-',  lw=2.0, label='Hydro (no helicity)')
    ax.loglog(f_array, results[(zeta,'A')] * 1e20,   'b-.', lw=1.8, label='Model A (MHD)')
    ax.loglog(f_array, results[(zeta,'B')] * 1e20,   'r--', lw=1.8, label='Model B (MHD)')

    ax.axvline(f_peak_I,  color='gray',   ls=':', lw=1.2, label=rf'$f_{{I}}$={f_peak_I:.1e} Hz')
    ax.axvline(f_peak_II, color='purple', ls=':', lw=1.2, label=rf'$f_H$={f_peak_II:.1e} Hz')

    ax.set_xlabel('f  [Hz]')
    ax.set_xlim(f_array[0], f_array[-1])
    ax.set_title(rf'$\zeta_* = {zeta}$,  $T_*=100$ GeV,  $\alpha={alpha_fig}$,  $\beta/H_*={beta_fig:.0f}$')
    ax.legend(fontsize=10)
    ax.grid(True, which='both', alpha=0.3)

axes[0].set_ylabel(r'$h_C(f) \times 10^{20}$')
fig.suptitle('GW strain amplitude — PhysRevD 78, 123006 (2008), Fig. 1 reproduction\n'
             r'(Eq. 51 implementation, $T_*=100$ GeV)', y=1.01)
plt.tight_layout()
plt.savefig('kahniashvili2008_fig1_reproduction.pdf', bbox_inches='tight')
plt.show()
print("Saved kahniashvili2008_fig1_reproduction.pdf")


In [ ]:
T_fig2  = 250.0
g_fig2  = 100.0
tp_fig2 = turb_params(alpha_fig, beta_fig, T_fig2, g_star=g_fig2)
f_array2 = np.geomspace(1e-5, 1e-2, 80)

f_peak_I_2  = tp_fig2['k0'] * tp_fig2['M'] / (2*np.pi) * tp_fig2['fH_Hz']
f_peak_II_2 = tp_fig2['fH_Hz']
print(f"T_* = 250 GeV,  f_H = {tp_fig2['fH_Hz']:.3e} Hz")

print("Computing unmagnetized (T=250 GeV)...")
hC_unmag2 = h_C_unmagnetized(f_array2, tp_fig2)

results2 = {}
for zeta in zeta_vals:
    for model in ['A', 'B']:
        print(f"Computing T=250 GeV, zeta={zeta}, Model {model}...")
        results2[(zeta, model)] = h_C_spectrum(f_array2, tp_fig2, zeta, model=model)
        print(f"  peak h_C = {results2[(zeta,model)].max():.3e}")


In [ ]:
fig2, axes2 = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax_idx, zeta in enumerate(zeta_vals):
    ax = axes2[ax_idx]

    ax.loglog(f_array2, hC_unmag2 * 1e20,             'k-',  lw=2.0, label='Hydro (no helicity)')
    ax.loglog(f_array2, results2[(zeta,'A')] * 1e20,  'b-.', lw=1.8, label='Model A (MHD)')
    ax.loglog(f_array2, results2[(zeta,'B')] * 1e20,  'r--', lw=1.8, label='Model B (MHD)')

    ax.axvline(f_peak_I_2,  color='gray',   ls=':', lw=1.2)
    ax.axvline(f_peak_II_2, color='purple', ls=':', lw=1.2)

    ax.set_xlabel('f  [Hz]')
    ax.set_xlim(f_array2[0], f_array2[-1])
    ax.set_title(rf'$\zeta_* = {zeta}$,  $T_*=250$ GeV,  $\alpha={alpha_fig}$,  $\beta/H_*={beta_fig:.0f}$')
    ax.legend(fontsize=10)
    ax.grid(True, which='both', alpha=0.3)

axes2[0].set_ylabel(r'$h_C(f) \times 10^{20}$')
fig2.suptitle('GW strain amplitude — PhysRevD 78, 123006 (2008), Fig. 2 reproduction\n'
              r'($T_*=250$ GeV)', y=1.01)
plt.tight_layout()
plt.savefig('kahniashvili2008_fig2_reproduction.pdf', bbox_inches='tight')
plt.show()
print("Saved kahniashvili2008_fig2_reproduction.pdf")


## Summary of key equations implemented

| Quantity | Eq. # | Implementation |
|---|---|---|
| $H_*$ | (16) | `H_star_Hz` |
| $a_*/a_0$ | (11) | `scale_factor_ratio` |
| $f_H$ | (53) | `f_H_Hz` |
| $\kappa(\alpha)$ | (13) | `kappa` |
| $v_b(\alpha)$ | (14) | `v_bubble` |
| $v_0(\alpha)$ | (15) | `v_eddy` |
| $k_S$ Model A | (35) | `k_S_modelA` |
| $H_{ijij}^{(II)}$ Model A | (45) | `H_ijij_modelA` |
| $k_S$ Model B | (41) | `k_S_modelB` |
| $H_{ijij}^{(II)}$ Model B | (47) | `H_ijij_modelB` |
| $H_{ijij}^{(I)}$ direct cascade | (43) | `H_ijij_direct` |
| $\Omega_{\rm GW}(\omega)$ | (49) | `Omega_GW` |
| $h_C(f)$ | (50) | `h_C` |

**Physical picture:**
- The GW spectrum has **two peaks**: the first (higher-$f$) from direct-cascade hydrodynamic turbulence at $f^{(I)} \propto k_0 M \propto (M/v_b)(\beta/H_*) f_H$, and the second (lower-$f$) from inverse-cascade MHD turbulence at $f^{(II)} = f_H$ (the Hubble frequency).
- For nonzero helicity the inverse-cascade peak can dominate, producing a detectable signal at lower frequencies.
- Models A and B give similar spectra (within a factor of $\sim 2$), confirming robustness.